<a href="https://colab.research.google.com/github/apar-tech/rag-chatbot/blob/main/phase4/ragas_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Networking RAG - Phase 4 Evaluation

This notebook evaluates the improved Networking RAG system developed in Phase 4. The evaluation is performed using the same test questions and RAGAS metrics used previously, allowing comparison between the baseline and improved retrieval pipelines.

In [ ]:
# Install required libraries

try:

    !pip install -r requirements.txt

    print(
        "Dependencies installed successfully!"
    )

except Exception as e:

    print(
        "Installation Error:"
    )

    print(e)

In [ ]:
# Import required libraries

import os
import time
import pickle
import chromadb
import pandas as pd

from datasets import Dataset

from google import genai
from google.colab import userdata

from groq import Groq

print(
    "Libraries imported successfully!"
)

Libraries imported successfully!


In [ ]:
# Import RAGAS

try:

    import ragas

    print(
        "RAGAS imported successfully!"
    )

    print(
        "Version:",
        ragas.__version__
    )

except Exception as e:

    print(e)

RAGAS imported successfully!
Version: 0.4.3


In [ ]:
# Initialize Gemini

try:

    gemini_client = genai.Client(
        api_key=userdata.get(
            "geminikey"
        )
    )

    print(
        "Gemini initialized!"
    )

except Exception as e:

    print(e)

Gemini initialized!


In [ ]:
# Initialize Groq

try:

    groq_client = Groq(
        api_key=userdata.get(
            "groqkey"
        )
    )

    print(
        "Groq initialized!"
    )

except Exception as e:

    print(e)

Groq initialized!


In [ ]:
# Load ChromaDB

try:

    chroma_client = chromadb.PersistentClient(
        path="./networking_chromadb_phase4"
    )

    collection = (
        chroma_client.get_or_create_collection(
            name="digital_docs_phase4"
        )
    )

    print(
        "Records:",
        collection.count()
    )

except Exception as e:

    print(e)

Records: 0


In [ ]:
# Load saved chunks and embeddings

try:

    with open(
        "improved_chunks.pkl",
        "rb"
    ) as f:

        improved_chunks = pickle.load(f)

    with open(
        "improved_embeddings.pkl",
        "rb"
    ) as f:

        improved_embeddings = pickle.load(f)

    if collection.count() == 0:

        collection.add(

            ids=[
                f"chunk_{i}"
                for i in range(
                    len(improved_chunks)
                )
            ],

            documents=
            improved_chunks,

            embeddings=
            improved_embeddings
        )

    print(
        "Collection ready!"
    )

    print(
        "Records:",
        collection.count()
    )

except Exception as e:

    print(e)

Collection ready!
Records: 306


In [ ]:
# Load evaluation questions

try:

    def load_evaluation_data(
        path="array.txt"
    ):

        globals_dict = {}

        with open(
            path,
            "r"
        ) as f:

            exec(
                f.read(),
                globals_dict
            )

        return (

            globals_dict[
                "test_questions"
            ],

            globals_dict[
                "ground_truths"
            ]

        )

    test_questions, ground_truths = (

        load_evaluation_data()

    )

    # Keep only first 5 questions

    test_questions = (
        test_questions[:5]
    )

    ground_truths = (
        ground_truths[:5]
    )

    print(
        "Questions:",
        len(test_questions)
    )

    print(
        "Ground Truths:",
        len(ground_truths)
    )

except Exception as e:

    print(
        "Loading Error:"
    )

    print(e)

Questions: 5
Ground Truths: 5


In [ ]:
# Install BM25 library

try:

    !pip install -q rank-bm25

    print(
        "BM25 library installed successfully!"
    )

except Exception as e:

    print(
        "Installation Error:"
    )

    print(e)

BM25 library installed successfully!


In [ ]:
# Create BM25 index

try:

    from rank_bm25 import BM25Okapi

    tokenized_chunks = [

        chunk.lower().split()

        for chunk in improved_chunks

    ]

    bm25 = BM25Okapi(
        tokenized_chunks
    )

    print(
        "BM25 index created successfully!"
    )

    print(
        "Indexed Chunks:",
        len(tokenized_chunks)
    )

except Exception as e:

    print(e)

BM25 index created successfully!
Indexed Chunks: 306


In [ ]:
# RAG Query Function

try:

    def ask_network_question(
        question
    ):

        query_response = (
            gemini_client.models.embed_content(
                model="models/gemini-embedding-001",
                contents=question
            )
        )

        query_embedding = (
            query_response.embeddings[0].values
        )

        vector_results = (
            collection.query(
                query_embeddings=[
                    query_embedding
                ],
                n_results=5
            )
        )

        vector_chunks = (
            vector_results["documents"][0]
        )

        tokenized_query = (
            question.lower().split()
        )

        bm25_scores = (
            bm25.get_scores(
                tokenized_query
            )
        )

        bm25_indices = (
            sorted(
                range(
                    len(bm25_scores)
                ),
                key=lambda i:
                bm25_scores[i],
                reverse=True
            )[:5]
        )

        bm25_chunks = [

            improved_chunks[i]

            for i in bm25_indices

        ]

        retrieved_chunks = []

        for chunk in (
            vector_chunks +
            bm25_chunks
        ):

            if chunk not in retrieved_chunks:

                retrieved_chunks.append(
                    chunk
                )

        retrieved_chunks = (
            retrieved_chunks[:5]
        )

        context = "\n\n".join(
            retrieved_chunks
        )

        prompt = f"""
You are a networking assistant.

Answer only using the provided context.

Context:
{context}

Question:
{question}
"""

        response = (
            groq_client.chat.completions.create(
                model=
                "llama-3.3-70b-versatile",
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ]
            )
        )

        return {

            "answer":
            response.choices[0]
            .message.content,

            "contexts":
            retrieved_chunks
        }

    print(
        "Function created successfully!"
    )

except Exception as e:

    print(e)

Function created successfully!


In [ ]:
# Test hybrid search

try:

    result = (
        ask_network_question(
            "What is machine laerning?"
        )
    )

    print(
        result["answer"]
    )

except Exception as e:

    print(e)

Machine learning is the study of programs that can improve their performance on a given task automatically. It is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without being explicitly programmed.


In [ ]:
# Generate answers for evaluation questions

try:

    generated_answers = []

    retrieved_contexts = []

    for i, question in enumerate(
        test_questions
    ):

        print(
            f"Processing {i+1}/{len(test_questions)}"
        )

        result = (
            ask_network_question(
                question
            )
        )

        generated_answers.append(
            result["answer"]
        )

        retrieved_contexts.append(
            result["contexts"]
        )

        if (
            (i + 1) % 2 == 0
            and
            (i + 1) < len(
                test_questions
            )
        ):

            print(
                "Rate limit protection activated. Waiting 30 seconds..."
            )

            time.sleep(30)

    print(
        "Generation completed!"
    )

except Exception as e:

    print(
        "Generation Error:"
    )

    print(e)

Processing 1/5
Processing 2/5
Rate limit protection activated. Waiting 30 seconds...
Processing 3/5
Processing 4/5
Rate limit protection activated. Waiting 30 seconds...
Processing 5/5
Generation completed!


In [ ]:
# Create Evaluation Dataset

try:

    evaluation_dataset = (
        Dataset.from_dict(
            {
                "question":
                test_questions,

                "answer":
                generated_answers,

                "contexts":
                retrieved_contexts,

                "ground_truth":
                ground_truths
            }
        )
    )

    print(
        evaluation_dataset
    )

except Exception as e:

    print(
        "Dataset Error:"
    )

    print(e)

Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 5
})


In [ ]:
# Configure evaluator models

try:

    from langchain_groq import ChatGroq

    from langchain_google_genai import (
        GoogleGenerativeAIEmbeddings
    )

    evaluator_llm = (
        ChatGroq(
            model=
            "llama-3.3-70b-versatile",

            api_key=
            userdata.get(
                "groqkey"
            )
        )
    )

    evaluator_embeddings = (
        GoogleGenerativeAIEmbeddings(
            model=
            "models/gemini-embedding-001",

            google_api_key=
            userdata.get(
                "geminikey"
            )
        )
    )

    print(
        "Evaluator configured successfully!"
    )

except Exception as e:

    print(
        "Configuration Error:"
    )

    print(e)

Evaluator configured successfully!


In [ ]:
# Import RAGAS metrics

try:

    from ragas import evaluate

    from ragas.metrics import (
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    )

    print(
        "Metrics imported successfully!"
    )

except Exception as e:

    print(
        "Import Error:"
    )

    print(e)

Metrics imported successfully!


/tmp/ipykernel_3857/4155784304.py:7: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/tmp/ipykernel_3857/4155784304.py:7: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/tmp/ipykernel_3857/4155784304.py:7: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
/tmp/ipykernel_3857/4155784304.py:7: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed

In [ ]:
from ragas.run_config import RunConfig

run_config = RunConfig(
    timeout=300
)

In [ ]:
# Evaluate improved RAG pipeline

try:

    results = evaluate(

        evaluation_dataset,

        metrics=[

            faithfulness,

            answer_relevancy,

            context_precision,

            context_recall

        ],

        llm=evaluator_llm,

        embeddings=
        evaluator_embeddings,
        run_config=run_config
    )

    print(
        "Evaluation completed successfully!"
    )

    print(
        results
    )

except Exception as e:

    print(
        "Evaluation Error:"
    )

    print(e)

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Evaluation completed successfully!
{'faithfulness': 1.0000, 'answer_relevancy': 0.7894, 'context_precision': 0.8078, 'context_recall': 0.6000}


In [ ]:
# Save evaluation results

try:

    results_df = (
        results.to_pandas()
    )

    print(
        results_df
    )

    results_df.to_csv(

        "phase4_ragas_results.csv",

        index=False

    )

    print(
        "Results saved successfully!"
    )

except Exception as e:

    print(e)

                                          user_input  \
0                   What is Artificial Intelligence?   
1                     What are the main types of AI?   
2  What is the difference between AI and Machine ...   
3       What are some real-world applications of AI?   
4  What are the advantages of Artificial Intellig...   

                                  retrieved_contexts  \
0  [Existential risk\nRecent public debates in ar...   
1  [Narrow vs. general AI\nAI researchers are div...   
2  [Machine learning (ML) is a field of study in ...   
3  [Applications\nAI and machine learning technol...   
4  [Ethics\nAI has potential benefits and potenti...   

                                            response  \
0  Artificial intelligence (AI) is the capability...   
1  The main types of AI can be categorized into t...   
2  Machine learning (ML) grew out of the quest fo...   
3  Some real-world applications of AI include:\n\...   
4  According to Demis Hassabis of DeepMind, AI